# Orquestador del Piloto de Automatización ICBF

Este notebook centraliza todo el proceso de consolidación, validación y auditoría de los datos de la modalidad **Integrales**. 

### ⚠️ Requisitos e Instrucciones para el Usuario

Para que este proceso funcione correctamente, asegúrese de cumplir con lo siguiente:

1.  **Archivo Maestro**: El archivo de "Fuente de Verdad" nacional (e.g., `UDS_28042026_10AM.xlsx`) debe estar ubicado en la carpeta `./master_data/`.
2.  **Archivos Regionales**: Los archivos Excel recibidos de las regionales deben colocarse en su subcarpeta correspondiente dentro de `./shared_folders/` (e.g., los archivos de Boyacá van en la carpeta `BOYACA`).
3.  **Librerías de Python**: Debe tener instaladas las librerías `pandas` y `openpyxl` (`pip install pandas openpyxl`).
4.  **Ejecución**: Ejecute las celdas en orden. El sistema detectará automáticamente nuevos archivos y actualizará el reporte de auditoría.

### Objetivo
Automatizar la preparación de carpetas, la recolección de datos regionales y la generación de una **Tabla Maestra de Auditoría** para garantizar la transparencia y confiabilidad de la información frente a los stakeholders.

---

## 0. Configuración General

En esta celda se definen los parámetros clave que permiten que el sistema sea flexible si cambian los nombres de las hojas o columnas en el futuro.

In [1]:
import os
import pandas as pd
import datetime
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

CONFIG = {
    'MASTER_SHEET': 'UDS_28042026_10AM',
    'REGIONAL_SHEET_KEYWORD': 'ZONIFICAC',
    'MASTER_COL_REGIONAL': 'Regional UDS',
    'MASTER_COL_CUPOS': 'Cupos UDS',
    'REGIONAL_COL_HEADER_SEARCH': 'Regional UDS',
    'REGIONAL_COL_REGIONAL': 'Regional UDS',
    'REGIONAL_COL_CUPOS': 'Cupos',
    'BASE_DIR': r'D:\ICBF\cost-tracking\pilot_automation_v1'
}

MASTER_FILE = os.path.join(CONFIG['BASE_DIR'], 'master_data', 'UDS_28042026_10AM.xlsx')
SHARED_FOLDERS = os.path.join(CONFIG['BASE_DIR'], 'shared_folders')
OUTPUT_DIR = os.path.join(CONFIG['BASE_DIR'], 'output')
AUDIT_FILE = os.path.join(OUTPUT_DIR, 'PILOT_AUDIT_MASTER.csv')
CONSOLIDATED_FILE = os.path.join(OUTPUT_DIR, 'CONSOLIDATED_DATA.csv')

def clean_name(name):
    return str(name).strip().upper().replace('Á', 'A').replace('É', 'E').replace('Í', 'I').replace('Ó', 'O').replace('Ú', 'U').replace('Ñ', 'N')

print("Configuración cargada correctamente.")

Configuración cargada correctamente.


## 1. Inicialización del Entorno

Esta celda crea la estructura de carpetas necesaria para el piloto. Es **inteligente**: solo crea las carpetas si no existen.

In [3]:
def setup_environment():
    if not os.path.exists(SHARED_FOLDERS):
        os.makedirs(SHARED_FOLDERS)
    
    print("Identificando regionales desde el Maestro...")
    df_master = pd.read_excel(MASTER_FILE, sheet_name=CONFIG['MASTER_SHEET'])
    regionals = df_master[CONFIG['MASTER_COL_REGIONAL']].dropna().unique()
    
    count = 0
    for reg in regionals:
        reg_path = os.path.join(SHARED_FOLDERS, clean_name(reg))
        if not os.path.exists(reg_path):
            os.makedirs(reg_path)
            count += 1
    
    print(f"Entorno listo. Se crearon {count} nuevas carpetas regionales.")

setup_environment()

Identificando regionales desde el Maestro...
Entorno listo. Se crearon 33 nuevas carpetas regionales.


## 1.5. Generación Automática de Plantillas (Motor COM)

Esta celda toma el archivo maestro de **Zonificación** y la plantilla **MATRIZ SERVICIOS INTEGRALES**, y genera un archivo por cada regional.
Utiliza Microsoft Excel de forma nativa (`win32com`) para garantizar que:
1. Las contraseñas de protección sean manejadas automáticamente.
2. Las hojas y tablas queden completamente desbloqueadas para el usuario.
3. Las conexiones a datos externos y formatos se conserven intactos (sin alertas al abrir).
4. Los filtros residuales de Bolívar se limpien correctamente.

In [4]:
import pandas as pd
import win32com.client as win32
import os

def generate_templates():
    print("Iniciando Generación de Plantillas Nativas (COM)...", flush=True)
    master_zonificacion = os.path.join(CONFIG['BASE_DIR'], "master_data", "ZonificacIón_Primera_Infancia_Hasta_Oct.xlsx")
    template_file = os.path.join(CONFIG['BASE_DIR'], "master_data", "MATRIZ SERVICIOS INTEGRALES 2026 REGIONAL BOLIVAR_IP (2).xlsx")
    
    xl_master = pd.ExcelFile(master_zonificacion)
    
    MODALITY_CONFIG = {
        'Integrales': {
            'regional_idx': 1, 'cz_idx': 2, 'mun_idx': 3,
            'uds_code_idx': 4, 'uds_name_idx': 5,
            'modalidad_idx': 8, 'componente_idx': 9, 'cupos_idx': 10,
        },
        'Comunitarios': {
            'regional_idx': 0, 'cz_idx': 1, 'mun_idx': 2,
            'uds_code_idx': 4, 'uds_name_idx': 5,
            'modalidad_idx': 7, 'componente_idx': 6, 'cupos_idx': 10,
        }
    }
    
    excel = win32.Dispatch("Excel.Application")
    excel.Visible = False
    excel.DisplayAlerts = False
    excel.AskToUpdateLinks = False
    excel.EnableEvents = False
    
    try:
        for sheet_name, cfg in MODALITY_CONFIG.items():
            print(f"\n--- Modalidad: {sheet_name} ---", flush=True)
            df_master = pd.read_excel(xl_master, sheet_name=sheet_name, header=None)
            reg_col = cfg['regional_idx']
            df_data = df_master.dropna(subset=[reg_col])
            unique_regionals = [r for r in df_data[reg_col].unique() if "REGIONAL" not in str(r).upper() and "TIPO" not in str(r).upper()]
            
            for regional in unique_regionals:
                reg_clean = clean_name(regional)
                if reg_clean in ['', 'NAN']: continue
                
                output_path = os.path.join(SHARED_FOLDERS, reg_clean, f"MATRIZ_2026_{reg_clean}_{sheet_name.upper()}.xlsx")
                os.makedirs(os.path.dirname(output_path), exist_ok=True)
                print(f"  > Procesando {regional}...", flush=True)
                
                df_reg = df_data[df_data[reg_col] == regional]
                num_rows = len(df_reg)
                
                wb = excel.Workbooks.Open(template_file, UpdateLinks=0, ReadOnly=False)
                
                try:
                    wb.Unprotect("#APOLO1704*")
                    for s in wb.Sheets:
                        try: s.Unprotect("#APOLO1704*")
                        except: pass
                    
                    ws = wb.Sheets('Matriz (2)')
                    ws.Visible = -1
                    try: wb.Sheets('Matriz').Delete()
                    except: pass
                    try: ws.Name = 'Matriz'
                    except: pass
                    
                    for tbl in ws.ListObjects:
                        try:
                            if tbl.AutoFilter and tbl.AutoFilter.FilterMode:
                                tbl.AutoFilter.ShowAllData()
                        except: pass
                    
                    col_mapping = {
                        1: cfg['regional_idx'], 2: cfg['cz_idx'], 3: cfg['mun_idx'],
                        4: cfg['componente_idx'], 5: cfg['modalidad_idx'], 6: cfg['uds_name_idx'],
                        7: cfg['uds_code_idx'], 8: cfg['cupos_idx']
                    }
                    
                    block1_data, block2_data = [], []
                    for idx, row_data in df_reg.iterrows():
                        b1 = [row_data[col_mapping[c]] if c in col_mapping and pd.notna(row_data[col_mapping[c]]) else "" for c in range(1, 15)]
                        block1_data.append(b1)
                        block2_data.append([""] * 7)
                    
                    start_row = 3
                    end_row = start_row + num_rows - 1
                    
                    ws.Range(ws.Cells(start_row, 1), ws.Cells(end_row, 14)).Value = block1_data
                    ws.Range(ws.Cells(start_row, 16), ws.Cells(end_row, 22)).Value = block2_data
                    
                    clear_start = start_row + num_rows
                    if clear_start <= 3785:
                        ws.Range(f'A{clear_start}:N3785').ClearContents()
                        ws.Range(f'P{clear_start}:V3785').ClearContents()
                    
                    wb.SaveAs(output_path)
                finally:
                    wb.Close(False)
                
                print(f"    Guardado: {os.path.basename(output_path)}", flush=True)
                
    finally:
        excel.Quit()

# Ejecutar la generación de plantillas:
generate_templates()


Iniciando Generación de Plantillas Nativas (COM)...

--- Modalidad: Integrales ---
  > Procesando Amazonas...
    Guardado: MATRIZ_2026_AMAZONAS_INTEGRALES.xlsx
  > Procesando Antioquia...
    Guardado: MATRIZ_2026_ANTIOQUIA_INTEGRALES.xlsx
  > Procesando Arauca...
    Guardado: MATRIZ_2026_ARAUCA_INTEGRALES.xlsx
  > Procesando Atlántico...
    Guardado: MATRIZ_2026_ATLANTICO_INTEGRALES.xlsx
  > Procesando Bogota D.C....
    Guardado: MATRIZ_2026_BOGOTA D.C._INTEGRALES.xlsx
  > Procesando Bolívar...
    Guardado: MATRIZ_2026_BOLIVAR_INTEGRALES.xlsx
  > Procesando Boyacá...
    Guardado: MATRIZ_2026_BOYACA_INTEGRALES.xlsx
  > Procesando Caldas...
    Guardado: MATRIZ_2026_CALDAS_INTEGRALES.xlsx
  > Procesando Caquetá...
    Guardado: MATRIZ_2026_CAQUETA_INTEGRALES.xlsx
  > Procesando Casanare...
    Guardado: MATRIZ_2026_CASANARE_INTEGRALES.xlsx
  > Procesando Cauca...
    Guardado: MATRIZ_2026_CAUCA_INTEGRALES.xlsx
  > Procesando Cesar...
    Guardado: MATRIZ_2026_CESAR_INTEGRALES.xlsx

## 2. Motor de Reglas de Negocio (Validaciones)

Aquí definimos las funciones que validan la calidad del dato. Este sistema es **modular**: puede agregar nuevas reglas fácilmente.

In [ ]:
def rule_uds_count(df_reg, df_master_reg):
    if len(df_reg) != len(df_master_reg):
        return False, f"Diferencia en UDS: Maestro {len(df_master_reg)} vs Archivo {len(df_reg)}"
    return True, "OK"

def rule_cupo_sum(df_reg, df_master_reg):
    master_sum = df_master_reg[CONFIG['MASTER_COL_CUPOS']].sum()
    file_sum = pd.to_numeric(df_reg[CONFIG['REGIONAL_COL_CUPOS']], errors='coerce').sum()
    if abs(master_sum - file_sum) > 0.1:
        return False, f"Diferencia en Cupos: Maestro {master_sum} vs Archivo {file_sum}"
    return True, "OK"

VALIDATION_RULES = [
    ('Conteo_UDS', rule_uds_count),
    ('Suma_Cupos', rule_cupo_sum)
]

print("Reglas de negocio cargadas.")

## 3. Orquestación (Carga y Auditoría)

El "Harvester" escanea las carpetas, procesa los archivos y genera el reporte.

In [ ]:
def find_header(df_temp, keyword):
    for idx, row in df_temp.iterrows():
        if any(isinstance(val, str) and keyword.lower() in val.lower() for val in row.values):
            return idx
    return None

def run_orchestration():
    df_master_all = pd.read_excel(MASTER_FILE, sheet_name=CONFIG['MASTER_SHEET'])
    df_master_all['Regional_Clean'] = df_master_all[CONFIG['MASTER_COL_REGIONAL']].apply(clean_name)
    
    audit_results = []
    consolidated_dfs = []
    
    for reg in sorted(df_master_all['Regional_Clean'].unique()):
        reg_folder = os.path.join(SHARED_FOLDERS, reg)
        df_m_reg = df_master_all[df_master_all['Regional_Clean'] == reg]
        
        status = "PENDING"
        file_found = None
        val_logs = []
        
        if os.path.exists(reg_folder):
            files = [f for f in os.listdir(reg_folder) if f.endswith(('.xlsx', '.xlsm')) and not f.startswith('~$')]
            if files:
                file_found = sorted(files)[-1]
                path = os.path.join(reg_folder, file_found)
                
                try:
                    xls = pd.ExcelFile(path)
                    s_match = [s for s in xls.sheet_names if CONFIG['REGIONAL_SHEET_KEYWORD'] in s.upper()]
                    if s_match:
                        df_t = pd.read_excel(xls, sheet_name=s_match[0], nrows=15, header=None)
                        h_idx = find_header(df_t, CONFIG['REGIONAL_COL_HEADER_SEARCH'])
                        if h_idx is not None:
                            df_r = pd.read_excel(xls, sheet_name=s_match[0], header=h_idx)
                            col_r = [c for c in df_r.columns if CONFIG['REGIONAL_COL_REGIONAL'] in str(c)][0]
                            df_r = df_r.dropna(subset=[col_r])
                            
                            all_ok = True
                            for r_name, r_func in VALIDATION_RULES:
                                ok, msg = r_func(df_r, df_m_reg)
                                val_logs.append(f"[{r_name}]: {msg}")
                                if not ok: all_ok = False
                            
                            status = "VALIDATED" if all_ok else "ERROR"
                            consolidated_dfs.append(df_r)
                        else: status = "ERROR: Header Missing"
                    else: status = "ERROR: Sheet Missing"
                except Exception as e: status = f"ERROR: {str(e)}"
            else: status = "MISSING"
        
        audit_results.append({
            'Regional': reg,
            'Estado': status,
            'Archivo': file_found,
            'Log_Validacion': " | ".join(val_logs),
            'Fecha_Procesamiento': datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
        })
    
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
    pd.DataFrame(audit_results).to_csv(AUDIT_FILE, index=False)
    if consolidated_dfs:
        pd.concat(consolidated_dfs, ignore_index=True).to_csv(CONSOLIDATED_FILE, index=False)
    
    return pd.DataFrame(audit_results)

resumen = run_orchestration()
print("Orquestación completada.")

## 4. Reporte de Avances

Resumen visual del estado del piloto para los stakeholders.

In [ ]:
print("--- RESUMEN EJECUTIVO DE AVANCES ---")
print(f"Total Regionales: {len(resumen)}")
print(f"Validadas: {len(resumen[resumen['Estado'] == 'VALIDATED'])}")
print(f"Con Errores: {len(resumen[resumen['Estado'].str.contains('ERROR')])}")
print(f"Pendientes (Missing): {len(resumen[resumen['Estado'] == 'MISSING'])}")

print("\nDetalle de Errores:")
display(resumen[resumen['Estado'].str.contains('ERROR')][['Regional', 'Estado', 'Log_Validacion']])